# Inteligencia Computacional — Guía de trabajos prácticos 2

## Perceptrón multicapa

## Ejercicio 1

> Implemente el algoritmo de retropropagación para un perceptrón multicapa de forma que
> se pueda elegir libremente la cantidad de capas de la red y de neuronas en cada capa.
> Pruébelo entrenando una red de estructura apropiada para resolver el problema XOR, con
> sus particiones de entrenamiento y prueba correspondientes (datos de la Guía de Trabajos
> Prácticos 1).

In [ ]:
import numpy as np
from pathlib import Path
import pandas as pd

### La clase `PerceptronMulticapa`

Generaliza el perceptrón simple del TP1 a una red de $L$ capas entrenada con
retropropagación: la cantidad de capas y de neuronas por capa se elige libremente pasando
la lista `neuronas_por_capa` (por ejemplo `[2, 2, 1]`: 2 entradas, una capa oculta de 2
neuronas, 1 salida), tal como pide la consigna.

**Estado de la red**
- `pesos`: una lista de $L$ matrices, una por capa — `pesos[k]` conecta la capa $k$ con la
  $k+1$ y tiene forma `(neuronas_por_capa[k], neuronas_por_capa[k+1])`.
- `umbrales`: una lista de $L$ vectores, un umbral por neurona de cada capa (la de entrada
  no tiene).

Ambos se inicializan al azar en el rango -0,5 a 0,5, igual que en TP1.

**Activación.** Sigmoide bipolar, g(v) = 2 / (1 + e^-v) - 1, en el rango (-1, 1) — la misma
que usa el código de referencia de la cátedra —, con la derivada expresada en función de
la salida ya calculada: g'(v) = 0,5 (1 - y^2).

**Cómo predice**
- `propagar`: hace el forward pass capa por capa y devuelve la lista *completa* de
  activaciones (entrada incluida), no solo la salida final — hacen falta todas para el
  backward pass.
- `predecir`: se queda solo con la última, la salida de la red.

**Cómo aprende**
- `entrenar_un_patron`: retropropagación clásica. Calcula el delta de la capa de salida
  (error por derivada de la activación) y lo va empujando hacia atrás capa por capa —el
  delta de una capa oculta depende de los deltas de la capa siguiente multiplicados por
  los pesos que las conectan, la regla de la cadena—. Con esos deltas ajusta cada matriz
  de pesos en la dirección que reduce el error, igual que la regla del perceptrón de TP1
  pero ahora una vez por capa.
- `entrenar`: recorre todos los patrones, época tras época, contando tanto los patrones
  mal clasificados (error absoluto mayor a `tolerancia`, el mismo criterio que usa el
  código de referencia) como el error cuadrático total de la época —esto último no hace
  falta todavía, pero lo vamos a necesitar para las curvas de error del Ejercicio 3—.
  También guarda `historial`, una foto de los pesos y umbrales después de cada época,
  igual que `historial_pesos` en TP1: es lo que usamos abajo para animar cómo se mueve la
  región de decisión. Corta si una época entera queda sin errores.
- `probar`: por ahora asume una única neurona de salida y decide la clase por el signo,
  igual que el código de referencia. Para el Ejercicio 3 (Iris, 3 salidas) va a hacer
  falta otra regla de decisión.

In [ ]:
class PerceptronMulticapa:
    def __init__(self, neuronas_por_capa, tasa_aprendizaje=0.1, semilla=None):
        generador = np.random.default_rng(semilla)
        self.tasa_aprendizaje = tasa_aprendizaje
        self.pesos = [
            generador.uniform(-0.5, 0.5, (neuronas_por_capa[i], neuronas_por_capa[i + 1]))
            for i in range(len(neuronas_por_capa) - 1)
        ]
        self.umbrales = [
            generador.uniform(-0.5, 0.5, neuronas_por_capa[i + 1])
            for i in range(len(neuronas_por_capa) - 1)
        ]

    def activacion(self, entrada_neta):
        return 2 / (1 + np.exp(-entrada_neta)) - 1

    def derivada_activacion(self, salida):
        return 0.5 * (1 - salida ** 2)

    def propagar(self, entradas):
        activaciones = [entradas]
        for pesos_capa, umbral_capa in zip(self.pesos, self.umbrales):
            entrada_neta = activaciones[-1] @ pesos_capa + umbral_capa
            activaciones.append(self.activacion(entrada_neta))
        return activaciones

    def predecir(self, entradas):
        return self.propagar(entradas)[-1]

    def entrenar_un_patron(self, entradas, salida_deseada):
        activaciones = self.propagar(entradas)
        error_salida = salida_deseada - activaciones[-1]

        deltas = [error_salida * self.derivada_activacion(activaciones[-1])]
        for capa in range(len(self.pesos) - 1, 0, -1):
            delta_propagado = self.derivada_activacion(activaciones[capa]) * (self.pesos[capa] @ deltas[0])
            deltas.insert(0, delta_propagado)

        for capa in range(len(self.pesos)):
            self.pesos[capa] = self.pesos[capa] + self.tasa_aprendizaje * np.outer(activaciones[capa], deltas[capa])
            self.umbrales[capa] = self.umbrales[capa] + self.tasa_aprendizaje * deltas[capa]

        return error_salida

    def _copiar_estado(self):
        return ([w.copy() for w in self.pesos], [b.copy() for b in self.umbrales])

    def entrenar(self, entradas_entrenamiento, salidas_deseadas, maximo_epocas, tolerancia=0.1):
        errores_por_epoca = []
        error_cuadratico_por_epoca = []
        historial = [self._copiar_estado()]

        for numero_de_epoca in range(maximo_epocas):
            errores_en_esta_epoca = 0
            error_cuadratico_total = 0.0

            for entradas, salida_deseada in zip(entradas_entrenamiento, salidas_deseadas):
                error = self.entrenar_un_patron(entradas, salida_deseada)
                error_cuadratico_total += np.sum(error ** 2)
                if np.any(np.abs(error) > tolerancia):
                    errores_en_esta_epoca += 1

            errores_por_epoca.append(errores_en_esta_epoca)
            error_cuadratico_por_epoca.append(error_cuadratico_total)
            historial.append(self._copiar_estado())

            if errores_en_esta_epoca == 0:
                break

        return errores_por_epoca, error_cuadratico_por_epoca, historial

    def probar(self, entradas_prueba, salidas_deseadas_prueba):
        cantidad_patrones = len(entradas_prueba)
        cantidad_errores = 0

        for entradas, salida_deseada in zip(entradas_prueba, salidas_deseadas_prueba):
            salida_predicha = 1 if self.predecir(entradas)[0] >= 0 else -1
            if salida_predicha != salida_deseada:
                cantidad_errores += 1

        porcentaje_aciertos = 100 * (cantidad_patrones - cantidad_errores) / cantidad_patrones
        return porcentaje_aciertos, cantidad_errores

In [ ]:
from src.data_loader import cargar_patrones

DIRECTORIO_DATASET = Path("Dataset")

tasa_aprendizaje = 0.1
semilla = 42
maximo_epocas = 1000
tolerancia = 0.1

entradas_xor, salidas_deseadas_xor = cargar_patrones(DIRECTORIO_DATASET / "XOR_trn.csv")
entradas_xor_prueba, salidas_deseadas_xor_prueba = cargar_patrones(DIRECTORIO_DATASET / "XOR_tst.csv")

mlp_xor = PerceptronMulticapa(
    neuronas_por_capa=[entradas_xor.shape[1], 4, 1],
    tasa_aprendizaje=tasa_aprendizaje,
    semilla=semilla,
)

errores_por_epoca_xor, error_cuadratico_por_epoca_xor, historial_xor = mlp_xor.entrenar(
    entradas_xor,
    salidas_deseadas_xor,
    maximo_epocas=maximo_epocas,
    tolerancia=tolerancia,
)

porcentaje_aciertos_xor, cantidad_errores_xor = mlp_xor.probar(
    entradas_xor_prueba, salidas_deseadas_xor_prueba
)

convergio = errores_por_epoca_xor[-1] == 0

pd.DataFrame(
    [
        ("Patrones de entrenamiento", len(salidas_deseadas_xor)),
        ("Patrones de prueba", len(salidas_deseadas_xor_prueba)),
        ("Arquitectura", "2 -> 4 -> 1"),
        ("Tasa de aprendizaje", tasa_aprendizaje),
        ("Criterio de corte", "cero errores" if convergio else "máximo de épocas"),
        ("Épocas utilizadas", f"{len(errores_por_epoca_xor)} de {maximo_epocas}"),
        ("Error cuadrático final", round(error_cuadratico_por_epoca_xor[-1], 4)),
        ("Aciertos en prueba",
         f"{porcentaje_aciertos_xor:.1f} % ({cantidad_errores_xor} errores sobre {len(salidas_deseadas_xor_prueba)})"),
    ],
    columns=["Métrica", "Valor"],
)

Igual que en TP1, conviene ver no solo el caso que funciona sino también uno que se traba, para
entender qué papel juega la arquitectura. Con solo **2 neuronas ocultas** (la mínima cantidad
capaz de resolver XOR en teoría) la red de arriba, con esta misma semilla, queda estancada: en
1000 épocas nunca deja de equivocarse en los 2000 patrones de entrenamiento y el acierto en
prueba cae a apenas ~56 %, prácticamente azar. Con **4 neuronas ocultas** converge en pocas
épocas. Entrenamos ambas y animamos cómo se mueve la región de decisión en cada caso.

In [ ]:
from src.graficos import graficar_y_animar_regiones

DIRECTORIO_GRAFICOS = Path("Graficos")

mlp_xor_no_converge = PerceptronMulticapa(
    neuronas_por_capa=[entradas_xor.shape[1], 2, 1],
    tasa_aprendizaje=tasa_aprendizaje,
    semilla=semilla,
)

errores_por_epoca_xor_2, error_cuadratico_por_epoca_xor_2, historial_xor_2 = mlp_xor_no_converge.entrenar(
    entradas_xor,
    salidas_deseadas_xor,
    maximo_epocas=200,
    tolerancia=tolerancia,
)

pd.DataFrame(
    [
        ("Épocas utilizadas", f"{len(errores_por_epoca_xor_2)} de 200"),
        ("Errores en la última época", f"{errores_por_epoca_xor_2[-1]} de {len(salidas_deseadas_xor)}"),
    ],
    columns=["Métrica", "Valor"],
)

In [ ]:
graficar_y_animar_regiones(
    [
        (entradas_xor, salidas_deseadas_xor, historial_xor, "XOR (4 ocultas)"),
        (entradas_xor, salidas_deseadas_xor, historial_xor_2, "XOR (2 ocultas)"),
    ],
    ruta_gif=DIRECTORIO_GRAFICOS / "xor_entrenamiento.gif",
)

### Conclusión del Ejercicio 1

_Pendiente._

## Ejercicio 2

> Utilice para entrenamiento y prueba los conjuntos de datos `concent_trn.csv` y
> `concent_tst.csv`, que consisten en dos clases distribuidas en forma concéntrica.
> Determine la estructura de una red de tipo perceptrón multicapa que resulte más
> apropiada para resolver este problema. Represente gráficamente, con diferentes colores,
> el resultado de la clasificación realizada por el perceptrón multicapa.

In [ ]:
# TODO: implementación del Ejercicio 2

### Conclusión del Ejercicio 2

_Pendiente._

## Ejercicio 3

> Iris es el género de una planta herbácea con flores que se utilizan en decoración.
> Dentro de este género existen muy diversas especies, entre las que se han estudiado:
> *Iris setosa*, *Iris versicolor* e *Iris virginica*. Estas tres especies pueden
> distinguirse según las dimensiones de sus pétalos y sépalos. Un grupo de investigadores
> ha recopilado la información correspondiente a las longitudes y anchos de los pétalos y
> sépalos de 50 plantas de cada especie. En el archivo `iris81_trn.csv` se encuentra el
> conjunto de entrenamiento, y en `iris81_tst.csv` el de prueba, generado a partir de estas
> mediciones (en cm), junto con un código binario que indica la clase de cada muestra
> (especie) reconocida por el grupo de investigadores ($[-1,-1,1]$ = setosa,
> $[-1,1,-1]$ = versicolor, $[1,-1,-1]$ = virginica).
>
> Determine la estructura óptima de un perceptrón multicapa para resolver este problema.
> Explore cómo varía el desempeño al usar distintas tasas de aprendizaje, y para cada caso
> grafique las curvas de error cuadrático total y error de clasificación en función de las
> épocas de entrenamiento.

In [ ]:
# TODO: implementación del Ejercicio 3

### Conclusión del Ejercicio 3

_Pendiente._